In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os
import platform
import warnings
warnings.filterwarnings('ignore')

# -----------------------------------------------------------
# [기본 설정] 맥(Mac) / 윈도우 한글 폰트 자동 설정
# -----------------------------------------------------------
if platform.system() == 'Darwin':
    plt.rc('font', family='AppleGothic')
else:
    plt.rc('font', family='Malgun Gothic')
plt.rcParams['axes.unicode_minus'] = False

# -----------------------------------------------------------
# [데이터 불러오기] 업로드된 실제 파일명으로 지정
# -----------------------------------------------------------
file_target = 'DB_Accidents_Aggregated_2014_2022.csv'

if not os.path.exists(file_target):
    print(f"🚨 '{file_target}' 파일을 찾을 수 없습니다. 현재 폴더에 파일이 있는지 확인해주세요.")
else:
    # 데이터 로드 (UTF-8 인코딩을 우선 적용하고, 실패 시 CP949 적용)
    try:
        df_accidents = pd.read_csv(file_target, encoding='utf-8')
    except UnicodeDecodeError:
        df_accidents = pd.read_csv(file_target, encoding='cp949')

    # 컬럼명 앞뒤의 숨겨진 띄어쓰기 일괄 제거
    df_accidents.columns = df_accidents.columns.str.strip()
    
    # -----------------------------------------------------------
    # [데이터 전처리] 일반국도와 고속국도(고속도로)만 필터링
    # -----------------------------------------------------------
    try:
        df_compare = df_accidents[df_accidents['도로종류'].isin(['일반국도', '고속국도'])].copy()
    except KeyError:
        print("🚨 에러: '도로종류'라는 이름의 컬럼이 없습니다.")
        print("현재 데이터에 존재하는 컬럼명 목록입니다:", df_accidents.columns.tolist())
        raise  # 에러를 명확히 보여주기 위해 프로그램 중단

    # -----------------------------------------------------------
    # 📊 [그래프] 일반국도 vs 고속도로 사고 발생 추이 비교
    # -----------------------------------------------------------
    plt.figure(figsize=(12, 6))
    
    # 선 그래프 생성
    sns.lineplot(data=df_compare, x='연도', y='사고_건', hue='도로종류', 
                 palette=['#1f77b4', '#d62728'], marker='o', markersize=9, linewidth=3)
    
    # 각 점(마커) 위에 사고 건수 텍스트 표시
    for road_type in ['일반국도', '고속국도']:
        temp_df = df_compare[df_compare['도로종류'] == road_type]
        for _, row in temp_df.iterrows():
            # 고속도로는 점 아래에, 일반국도는 점 위에 글씨가 오도록 위치 조정
            y_offset = -15 if road_type == '고속국도' else 15
            plt.text(row['연도'], row['사고_건'] + y_offset, f"{int(row['사고_건'])}건", 
                     ha='center', fontsize=11, fontweight='bold',
                     color='#d62728' if road_type == '고속국도' else '#1f77b4')

    # 그래프 꾸미기 (데이터 연도 범위에 맞춰 제목 수정)
    plt.title('일반국도 vs 고속도로 사고 발생 추이 (2014~2022)', fontsize=16, fontweight='bold', pad=20)
    plt.xlabel('연도', fontsize=13)
    plt.ylabel('사고 건수 (건)', fontsize=13)
    
    # 범례 설정
    plt.legend(title='도로 종류', fontsize=12, title_fontsize=13)
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    
    # Y축 범위 넉넉하게 조정 (데이터 최대치 기준 1.2배 - 글씨 안 짤리게)
    plt.ylim(0, df_compare['사고_건'].max() * 1.2)
    
    plt.show()
    
    # -----------------------------------------------------------
    # 💡 [추가 분석] 치사율(사망자수 / 사고건수 * 100) 비교
    # -----------------------------------------------------------
    print("\n📌 [발표용 추가 인사이트: 치사율 비교]")
    
    try:
        df_compare['치사율(%)'] = (df_compare['사망_명'] / df_compare['사고_건']) * 100
        
        avg_lethality_national = df_compare[df_compare['도로종류'] == '일반국도']['치사율(%)'].mean()
        avg_lethality_highway = df_compare[df_compare['도로종류'] == '고속국도']['치사율(%)'].mean()
        
        print(f"- 일반국도 사고 평균 치사율: {avg_lethality_national:.1f}%")
        print(f"- 고속도로 사고 평균 치사율: {avg_lethality_highway:.1f}%")
    except KeyError as e:
        print(f"🚨 에러: {e} 컬럼을 찾을 수 없습니다. (치사율 계산 실패)")

🚨 'DB_Accidents_Aggregated_2014_2022.csv' 파일을 찾을 수 없습니다. 현재 폴더에 파일이 있는지 확인해주세요.


In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import platform
import warnings
warnings.filterwarnings('ignore')

# -----------------------------------------------------------
# [기본 설정] 맥/윈도우 폰트 깨짐 자동 방지
# -----------------------------------------------------------
if platform.system() == 'Darwin':
    plt.rc('font', family='AppleGothic')
else:
    plt.rc('font', family='Malgun Gothic')
plt.rcParams['axes.unicode_minus'] = False

# -----------------------------------------------------------
# [데이터 불러오기] 업로드된 상세현황 파일명으로 직접 지정
# -----------------------------------------------------------
file_acc = 'DB_Accidents_Detailed_2022_2024.csv'

if not os.path.exists(file_acc):
    print(f"🚨 '{file_acc}' 파일을 찾을 수 없습니다. 현재 폴더를 확인해주세요.")
else:
    # 인코딩 처리하며 파일 읽기
    try:
        df_acc = pd.read_csv(file_acc, encoding='utf-8')
    except UnicodeDecodeError:
        df_acc = pd.read_csv(file_acc, encoding='cp949')
        
    # 컬럼명 띄어쓰기 정리 (안전장치)
    df_acc.columns = df_acc.columns.str.strip()
        
    # -----------------------------------------------------------
    # [데이터 분석] 사고 원인별 발생 건수 집계 및 1% 미만 통합
    # -----------------------------------------------------------
    cause_counts = df_acc['원인'].value_counts()
    total_accidents = cause_counts.sum()
    
    # 1%에 해당하는 기준 건수 계산
    threshold = total_accidents * 0.01
    
    # 1% 이상인 주요 원인과 1% 미만인 소수 원인 분리
    main_causes = cause_counts[cause_counts >= threshold].copy()
    minor_causes_sum = cause_counts[cause_counts < threshold].sum()
    
    # 소수 원인들의 합을 '기타'에 더해주기 (기존에 '기타'가 있으면 합치고, 없으면 새로 생성)
    if minor_causes_sum > 0:
        if '기타' in main_causes.index:
            main_causes['기타'] += minor_causes_sum
        else:
            main_causes['기타'] = minor_causes_sum

    # 다시 크기순으로 정렬
    main_causes = main_causes.sort_values(ascending=False)
    
    # -----------------------------------------------------------
    # 📊 [그래프] 고속도로 교통사고 원인별 비중 파이 차트
    # -----------------------------------------------------------
    plt.figure(figsize=(10, 8))
    
    # 발표 핵심인 '졸음'과 '주시태만' 파이 조각을 살짝 튀어나오게 분리(강조)
    explode = [0.1 if idx in ['졸음', '주시태만'] else 0 for idx in main_causes.index]
    
    # 눈에 잘 띄는 색상 조합
    colors = plt.cm.Set3.colors

    # 파이 차트 생성
    plt.pie(main_causes.values, 
            labels=main_causes.index, 
            autopct='%1.1f%%', 
            startangle=140, 
            explode=explode,
            shadow=True,
            colors=colors,
            textprops={'fontsize': 13, 'weight': 'bold'})
            
    plt.title('고속도로 교통사고 원인별 발생 비중 (2022~2024)', fontsize=18, fontweight='bold', pad=20)
    plt.show()
    
    # -----------------------------------------------------------
    # 💡 [추가 인사이트 출력]
    # -----------------------------------------------------------
    sleep_related = main_causes.get('졸음', 0) + main_causes.get('주시태만', 0)
    ratio = (sleep_related / total_accidents) * 100
    
    print("-" * 50)
    print(f"💡 전체 고속도로 사고 중 '졸음' 및 '주시태만'이 차지하는 비중은 {ratio:.1f}% 입니다!")
    print(f"💡 (1% 미만의 세부 사고 원인들은 모두 '기타' 항목으로 깔끔하게 통합되었습니다.)")

🚨 'DB_Accidents_Detailed_2022_2024.csv' 파일을 찾을 수 없습니다. 현재 폴더를 확인해주세요.


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

sns.set_style("whitegrid")
plt.rcParams['font.family'] = 'AppleGothic'  # 윈도우 사용자라면 'Malgun Gothic'으로 변경
plt.rcParams['axes.unicode_minus'] = False   # 마이너스 기호 깨짐 방지

acc = pd.read_csv("DB_Accidents_Detailed_2022_2024.csv", encoding='utf-8')
she = pd.read_csv("DB_Shelter_Traffic_Hourly_2022_2024.csv", encoding='utf-8')

# 사고 데이터: 원인이 '졸음'인 데이터 추출 후 시간대(hour) 파싱
sleep_acc = acc[acc['원인'].astype(str).str.contains('졸음', na=False)].copy()
sleep_acc['hour'] = sleep_acc['사고시각'].str.split(':').str[0].astype(float)
acc_by_hour = sleep_acc.groupby('hour').size()

# 교통량 데이터: 시간대(hour) 파싱 후 합계 연산
she['총합계'] = pd.to_numeric(she['총합계'], errors='coerce')
she['hour'] = she['시간'].str.split(':').str[0].astype(float)
tr_by_hour = she.groupby('hour')['총합계'].sum()

# x축 (0시 ~ 23시) 배열 생성
x = np.arange(24)
y_acc = [acc_by_hour.get(h, 0) for h in x]
y_tr = [tr_by_hour.get(h, 0) for h in x]

# =========================================================
# 📊 [그래프] 졸음쉼터 이용률 vs 시간대별 사고 
# =========================================================
# 그래프가 1개로 줄었으므로 세로 길이를 12에서 6으로 조정
fig, ax1 = plt.subplots(figsize=(14, 6))

# 막대 그래프 (사고건수, 왼쪽 Y축)
ax1.bar(x, y_acc, color='#ff9999', label='졸음운전 사고건수 (좌측축)', edgecolor='black', linewidth=0.5)
ax1.set_ylabel('졸음운전 사고건수 (건)', fontsize=13, weight='bold', color='#e74c3c')
ax1.set_xticks(x)
ax1.set_xticklabels([f'{i}시' for i in x], fontsize=12)

# 꺾은선 그래프 (쉼터 이용량, 오른쪽 Y축)
ax2 = ax1.twinx()
ax2.plot(x, y_tr, color='#2c3e50', marker='o', markersize=8, linewidth=3, label='졸음쉼터 이용 통행량 (우측축)')
ax2.set_ylabel('쉼터 이용 통행량 (대)', fontsize=13, weight='bold', color='#2c3e50')

ax1.set_title('졸음쉼터 이용률 vs 시간대별 졸음사고 발생량 비교', fontsize=18, fontweight='bold', pad=15)

# 핵심 메시지 (주석 및 화살표)
ax1.annotate('야간/새벽:\n사고는 잦지만 쉼터 이용은 최저 수준', 
             xy=(3, y_acc[3]), xytext=(3, max(y_acc)*0.85),
             arrowprops=dict(facecolor='red', shrink=0.05, width=1.5, headwidth=8),
             ha='center', fontsize=13, fontweight='bold', color='red',
             bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="red", lw=1))

ax2.annotate('주간:\n이용량이 늘어나자 사고가 억제됨', 
             xy=(10, y_tr[10]), xytext=(10, max(y_tr)*0.55),
             arrowprops=dict(facecolor='blue', shrink=0.05, width=1.5, headwidth=8),
             ha='center', fontsize=13, fontweight='bold', color='blue',
             bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="blue", lw=1))

# 범례 통합
lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc='upper left', fontsize=12)

# =========================================================
# 마무리 및 이미지 저장
# =========================================================
plt.tight_layout()
plt.savefig('hourly_analysis_db.png', dpi=300)
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'DB_Accidents_Detailed_2022_2024.csv'